Celda 1 — conexión con BigQuery

In [1]:
from google.cloud import bigquery

client = bigquery.Client()

print("Proyecto conectado:", client.project)

c:\Users\Ainara Ximena\.conda\envs\mexico2030_env\lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.10.21) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
c:\Users\Ainara Ximena\.conda\envs\mexico2030_env\lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.10.21) which Google will stop supporting in new releases of google.cloud.bigquery once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.bigquery past that date.
  warnings.warn(message, FutureWarning)


Proyecto conectado: mexico2030analytics


In [2]:
import pandas as pd

In [3]:
silver_local = pd.read_csv(
    "../data/processed/matches_silver.csv",
    parse_dates=["date"]
)

print(silver_local.columns.tolist())
print("Registros Silver:", len(silver_local))

['match_id', 'match_key', 'date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']
Registros Silver: 49501


In [4]:
temp_csv = "../data/processed/silver_staging.csv"

silver_local.to_csv(
    temp_csv,
    index=False
)

print("CSV de staging creado correctamente.")
print("Registros:", len(silver_local))

CSV de staging creado correctamente.
Registros: 49501


Paso siguiente: cargar el CSV a BigQuery Staging

In [5]:
from google.cloud import bigquery

staging_table_id = "mexico2030analytics.silver.matches_staging"

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    schema=[
        bigquery.SchemaField("match_id", "INTEGER"),
        bigquery.SchemaField("match_key", "STRING"),
        bigquery.SchemaField("date", "DATE"),
        bigquery.SchemaField("home_team", "STRING"),
        bigquery.SchemaField("away_team", "STRING"),
        bigquery.SchemaField("home_score", "FLOAT"),
        bigquery.SchemaField("away_score", "FLOAT"),
        bigquery.SchemaField("tournament", "STRING"),
        bigquery.SchemaField("city", "STRING"),
        bigquery.SchemaField("country", "STRING"),
        bigquery.SchemaField("neutral", "BOOLEAN"),
    ],
)

with open(temp_csv, "rb") as f:
    job = client.load_table_from_file(
        f,
        staging_table_id,
        job_config=job_config
    )

job.result()

print("Staging cargado correctamente.")

Staging cargado correctamente.


prueba

In [6]:
query = """
SELECT
    COUNT(*) AS total,
    COUNT(match_key) AS con_match_key
FROM `mexico2030analytics.silver.matches_staging`
"""

result = client.query(query).result()

for row in result:
    print("Registros en staging:", row.total)
    print("Registros con match_key:", row.con_match_key)

Registros en staging: 49501
Registros con match_key: 49501


Perfecto. Staging está completo: 49,501 registros y 49,501 match_key no nulos.

Ahora viene una operación necesaria porque cuando agregamos match_key a silver.matches, la columna quedó inicialmente vacía para los registros que ya existían.

Paso 1: rellenar match_key en Silver

Vamos a utilizar match_id solo para este backfill, porque ambos lados ya contienen el mismo conjunto histórico. Después de esto, el proceso incremental usará match_key.

In [9]:
query = """
CREATE OR REPLACE TABLE `mexico2030analytics.silver.matches` AS
SELECT
    match_id,
    match_key,
    date,
    home_team,
    away_team,
    home_score,
    away_score,
    tournament,
    city,
    country,
    neutral
FROM `mexico2030analytics.silver.matches_staging`
"""

job = client.query(query)
job.result()

print("Silver reemplazado correctamente con match_key.")

Silver reemplazado correctamente con match_key.


Perfecto. Silver ya quedó reconstruido correctamente con match_key y sin utilizar DML.

Ahora necesitamos confirmar dos cosas antes de tocar Gold:

que silver.matches tenga los 49,501 registros;
que los 49,501 tengan match_key.

Esto es una comprobación puntual de la operación que acabamos de ejecutar.

In [10]:
query = """
SELECT
    COUNT(*) AS total,
    COUNT(match_key) AS con_match_key,
    COUNT(DISTINCT match_key) AS match_keys_unicos
FROM `mexico2030analytics.silver.matches`
"""

result = client.query(query).result()

for row in result:
    print("Registros Silver:", row.total)
    print("Registros con match_key:", row.con_match_key)
    print("match_key únicos:", row.match_keys_unicos)

Registros Silver: 49501
Registros con match_key: 49501
match_key únicos: 49501


Siguiente paso: preparar el MERGE

Ahora queremos que 004 haga la operación que realmente necesita el pipeline:

Silver local
     ↓
BigQuery
     ↓
MERGE por match_key
     ↓
si existe → no duplica
si no existe → INSERT

La ventaja de MERGE es que la lógica incremental queda en BigQuery, en lugar de descargar los 49,501 match_key cada vez para compararlos en Python.

In [14]:
query = """
CREATE OR REPLACE TABLE `mexico2030analytics.gold.fact_mexico_matches` AS

SELECT
    match_id,
    date AS match_date,
    EXTRACT(YEAR FROM date) AS year,

    CASE
        WHEN home_team = 'Mexico' THEN away_team
        ELSE home_team
    END AS opponent,

    tournament,

    CASE
        WHEN neutral = TRUE THEN 'Neutral'
        WHEN country = 'Mexico' THEN 'Home'
        ELSE 'Away'
    END AS venue_type,

    CASE
        WHEN home_team = 'Mexico' THEN home_score
        ELSE away_score
    END AS goals_for,

    CASE
        WHEN home_team = 'Mexico' THEN away_score
        ELSE home_score
    END AS goals_against,

    (
        CASE
            WHEN home_team = 'Mexico' THEN home_score
            ELSE away_score
        END
        -
        CASE
            WHEN home_team = 'Mexico' THEN away_score
            ELSE home_score
        END
    ) AS goal_difference,

    CASE
        WHEN
            (
                CASE
                    WHEN home_team = 'Mexico' THEN home_score
                    ELSE away_score
                END
            )
            >
            (
                CASE
                    WHEN home_team = 'Mexico' THEN away_score
                    ELSE home_score
                END
            )
        THEN 'Win'

        WHEN
            (
                CASE
                    WHEN home_team = 'Mexico' THEN home_score
                    ELSE away_score
                END
            )
            =
            (
                CASE
                    WHEN home_team = 'Mexico' THEN away_score
                    ELSE home_score
                END
            )
        THEN 'Draw'

        ELSE 'Loss'
    END AS result

FROM `mexico2030analytics.silver.matches`

WHERE home_team = 'Mexico'
   OR away_team = 'Mexico'
"""

job = client.query(query)
job.result()

print("Gold reconstruido correctamente.")

Gold reconstruido correctamente.


Perfecto. Gold ya fue reconstruido correctamente. Con esto, la parte estructural del pipeline ya está funcionando:

001 Ingestion
      ↓
002 Bronze
      ↓
003 Silver
      ↓
004 BigQuery
      ↓
Staging
      ↓
Silver BigQuery
      ↓
Gold

Ahora no quiero que sigamos agregando código sin comprobar el resultado final.

Última validación de Gold

In [15]:
query = """
SELECT
    COUNT(*) AS total,
    COUNT(DISTINCT match_id) AS partidos_unicos,
    MIN(match_date) AS primer_partido,
    MAX(match_date) AS ultimo_partido
FROM `mexico2030analytics.gold.fact_mexico_matches`
"""

result = client.query(query).result()

for row in result:
    print("Registros Gold:", row.total)
    print("Partidos únicos:", row.partidos_unicos)
    print("Primer partido:", row.primer_partido)
    print("Último partido:", row.ultimo_partido)

Registros Gold: 1008
Partidos únicos: 1008
Primer partido: 1923-01-01
Último partido: 2026-07-05


Perfecto. Gold también quedó consistente:

1,008 partidos de México.
1,008 match_id únicos.
Primer partido: 1923-01-01.
Último partido: 2026-07-05.

Con esto, la reconstrucción de las capas Silver y Gold está terminada.

Ahora sí vamos a hacer la prueba que realmente nos interesa: incrementalidad + idempotencia.

Ya tenemos el caso de prueba que habíamos preparado anteriormente:

Fecha:       2026-08-28
Local:       Mexico
Visitante:   Test Team
Marcador:    2 - 1
Torneo:      Test Tournament
match_id:    999999

Pero no vamos a modificar todavía tu results.csv real. Primero vamos a probar la lógica en memoria para no contaminar tus datos históricos.

Paso 1 — crear un registro nuevo de prueba

In [16]:
nuevo_partido = pd.DataFrame([{
    "match_id": 999999,
    "match_key": "2026-08-28|Mexico|Test Team|Test Tournament",
    "date": pd.Timestamp("2026-08-28"),
    "home_team": "Mexico",
    "away_team": "Test Team",
    "home_score": 2,
    "away_score": 1,
    "tournament": "Test Tournament",
    "city": "Test City",
    "country": "Mexico",
    "neutral": False
}])

print(nuevo_partido)

   match_id                                    match_key       date home_team  \
0    999999  2026-08-28|Mexico|Test Team|Test Tournament 2026-08-28    Mexico   

   away_team  home_score  away_score       tournament       city country  \
0  Test Team           2           1  Test Tournament  Test City  Mexico   

   neutral  
0    False  


Esto no toca BigQuery. Solo crea el partido ficticio en Pandas.

Paso 2 — comprobar que nuestra lógica lo identifica como nuevo

In [17]:
existing_keys = set(silver_local["match_key"])

new_matches = nuevo_partido[
    ~nuevo_partido["match_key"].isin(existing_keys)
]

print("Partidos existentes:", len(existing_keys))
print("Partidos nuevos detectados:", len(new_matches))
print("Nueva clave:", new_matches["match_key"].iloc[0])

Partidos existentes: 49501
Partidos nuevos detectados: 1
Nueva clave: 2026-08-28|Mexico|Test Team|Test Tournament


Esto demuestra la primera parte:

Un partido que no existe en Silver es detectado mediante match_key.

Pero todavía no hemos terminado

Después vamos a hacer la prueba inversa:

ejecución 1 → detecta 1 nuevo
ejecución 2 → detecta 0 nuevos

Eso es lo que nos permitirá demostrar la idempotencia.

Correcto. La primera prueba pasó exactamente como esperábamos.

Tenemos:

Partidos existentes:        49501
Partidos nuevos detectados: 1
Nueva clave:                2026-08-28|Mexico|Test Team|Test Tournament

Esto demuestra que match_key está funcionando como nuestra clave de detección.

Ahora vamos a hacer la segunda prueba: idempotencia. Aquí no vamos a subir nada a BigQuery todavía.

Paso 2 — ejecutar nuevamente la misma lógica

In [18]:
existing_keys = set(silver_local["match_key"])

new_matches_again = nuevo_partido[
    ~nuevo_partido["match_key"].isin(existing_keys)
]

print("Segunda ejecución")
print("Partidos existentes:", len(existing_keys))
print("Partidos nuevos detectados:", len(new_matches_again))

Segunda ejecución
Partidos existentes: 49501
Partidos nuevos detectados: 1


Aquí hay una precisión importante: como todavía no hemos agregado el partido ficticio a silver_local, esta segunda ejecución seguirá detectándolo como nuevo. Eso es correcto, pero todavía no demuestra idempotencia.

Por eso, después de comprobarlo, haremos la prueba correcta:

Silver actual
     ↓
agregamos Test Team
     ↓
ejecución → 1 nuevo
     ↓
Silver actualizado
     ↓
ejecución nuevamente → 0 nuevos

No quiero modificar silver_local hasta que veamos esta segunda salida.

Sí. Ese código es correcto para la prueba que estamos haciendo.

Pero hay un punto importante: como silver_local no ha cambiado y nuevo_partido tampoco se ha incorporado a él, esta segunda ejecución volverá a detectar 1 partido. Eso no demuestra todavía idempotencia.

La prueba correcta es simular que la primera ejecución ya incorporó el partido:

In [19]:
# Simulamos que el nuevo partido ya fue incorporado
silver_test = pd.concat(
    [silver_local, nuevo_partido],
    ignore_index=True
)

# Segunda ejecución
existing_keys = set(silver_test["match_key"])

new_matches_again = nuevo_partido[
    ~nuevo_partido["match_key"].isin(existing_keys)
]

print("Segunda ejecución")
print("Partidos existentes:", len(existing_keys))
print("Partidos nuevos detectados:", len(new_matches_again))

Segunda ejecución
Partidos existentes: 49502
Partidos nuevos detectados: 0


Esto sí demuestra la idempotencia

La secuencia es:

Estado original
49,501 partidos
       ↓
llega partido nuevo
       ↓
detecta 1 nuevo
       ↓
se incorpora
       ↓
49,502 partidos
       ↓
vuelve a llegar el mismo partido
       ↓
detecta 0 nuevos

Y lo importante es que la detección se basa en:

match_key

no en match_id.

Estado actual de la automatización

Ya tenemos comprobado:

Ingesta de datos → 001
Bronze → 002
Silver + limpieza + match_key → 003
Carga a BigQuery → 004
Reconstrucción de Gold → completada
Detección de nuevos registros → comprobada
Idempotencia → comprobada

Hay una precisión importante para la documentación del proyecto: por la restricción actual de BigQuery sin billing, no estamos usando MERGE/UPDATE para hacer incrementalidad directamente en BigQuery. La lógica incremental/idempotente está validada localmente y la materialización actual usa CREATE OR REPLACE.

Por tanto, no conviene afirmar en el CV o GitHub que tienes un incremental load con MERGE en BigQuery. La descripción técnicamente correcta sería algo como:

Implementé detección incremental e idempotente de nuevos partidos mediante match_key, con materialización de las capas Silver/Gold en BigQuery mediante reemplazo controlado debido a las restricciones de DML del free tier.

La siguiente etapa ya no es seguir haciendo pruebas de deduplicación. Podemos avanzar a automatizar el flujo completo para que 004_bigquery_incremental.ipynb ejecute el proceso de principio a fin con el dataset actualizado.